In [1]:
using JuMP, HiGHS, DataFrames

# 1. Define Sets / Indices
sources = ["Apron", "Terminal", "Cargo", "Access"]
destinations = ["Extension", "Pond", "Roads", "Parking", "Fire", "Industrial", "Perimeter"]

# 2. Define Data Structures (Option B)
available = Dict(
    "Apron"    => 660,
    "Terminal" => 301,
    "Cargo"    => 271,
    "Access"   => 99
)

needed = Dict(
    "Extension"  => 247,
    "Pond"       => 394,
    "Roads"      => 265,
    "Parking"    => 105,
    "Fire"       => 90,
    "Industrial" => 85,
    "Perimeter"  => 145
)

# Original DataFrame for distances
distance_df = DataFrame(
    :Deliver => ["Extension", "Pond", "Roads", "Parking", "Fire", "Industrial", "Perimeter"],
    :Apron => [26, 12, 10, 18, 11, 8, 20],
    :Terminal => [28, 14, 12, 20, 13, 10, 22],
    :Cargo => [20, 26, 20, 2, 6, 22, 18],
    :Access => [26, 10, 4, 16, 24, 14, 21]
)

# Convert distance table into a 2D Dictionary for clean indexing: cost[i, j]
cost = Dict(
    (row[:Deliver], src) => row[Symbol(src)] 
    for row in eachrow(distance_df) for src in sources
)

# 3. Build the JuMP Model
model = Model(HiGHS.Optimizer)

# Decision Variables: flow x[i, j] representing unit shipment from source i to destination j
@variable(model, x[i in sources, j in destinations] >= 0)

# Objective: Minimize total transportation cost (distance * volume)
@objective(model, Min, sum(cost[j, i] * x[i, j] for i in sources, j in destinations))

# Constraint 1: Supply capacity at each source site
@constraint(model, supply[i in sources], sum(x[i, j] for j in destinations) <= available[i])

# Constraint 2: Demand requirements at each destination site
@constraint(model, demand[j in destinations], sum(x[i, j] for i in sources) >= needed[j])

# 4. Solve and Extract Results
optimize!(model)

if termination_status(model) == OPTIMAL
    println("Total Minimum Cost: ", objective_value(model))
    println("\nOptimal Shipment Plan:")
    for i in sources, j in destinations
        val = value(x[i, j])
        if val > 0
            println("  Ship $val units from $i to $j")
        end
    end
end

Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: Apple Accelerate 
LP has 11 rows; 28 cols; 56 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [2e+00, 3e+01]
  Bound   [0e+00, 0e+00]
  RHS     [8e+01, 7e+02]
Presolving model
11 rows, 28 cols, 56 nonzeros 0s
11 rows, 28 cols, 56 nonzeros 0s
Presolve reductions: rows 11(-0); columns 28(-0); nonzeros 56(-0) - Not reduced
Problem not reduced by presolve: solving the LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Pr: 7(1331) 0.0s
         15     1.7592000000e+04 Pr: 0(0) 0.0s

Model status        : Optimal
Simplex   iterations: 15
Objective value     :  1.7592000000e+04
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00
Total Minimum Cost: 17592.0

Optimal Shipment Plan:
  Ship 319.0 units from

In [2]:
cost

Dict{Tuple{String, String}, Int64} with 28 entries:
  ("Extension", "Terminal")  => 28
  ("Parking", "Apron")       => 18
  ("Fire", "Apron")          => 11
  ("Extension", "Access")    => 26
  ("Pond", "Cargo")          => 26
  ("Roads", "Terminal")      => 12
  ("Parking", "Terminal")    => 20
  ("Industrial", "Terminal") => 10
  ("Roads", "Access")        => 4
  ("Parking", "Cargo")       => 2
  ("Fire", "Cargo")          => 6
  ("Pond", "Apron")          => 12
  ("Extension", "Cargo")     => 20
  ("Perimeter", "Apron")     => 20
  ("Perimeter", "Terminal")  => 22
  ("Perimeter", "Cargo")     => 18
  ("Perimeter", "Access")    => 21
  ("Pond", "Access")         => 10
  ("Industrial", "Cargo")    => 22
  ("Extension", "Apron")     => 26
  ("Roads", "Apron")         => 10
  ("Pond", "Terminal")       => 14
  ("Fire", "Access")         => 24
  ("Parking", "Access")      => 16
  ("Industrial", "Apron")    => 8
  ⋮                          => ⋮

In [3]:
objective_value(model)

17592.0